<a href="https://colab.research.google.com/github/tayyba6/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize pages using a simple directional score based on their March search-performance signals. Pages with stronger search exposure and weaker search performance receive higher priority for review.

The baseline uses **March impressions, CTR, and average position**. The score is intended for **prioritization and decision-support**, not as proof that a page needs a specific content change.

### Reason codes

* **HIGH_EXPOSURE** — the page has relatively high March impressions, so changes could affect a meaningful amount of search exposure.
* **LOW_CTR_OPPORTUNITY** — the page has relatively low CTR compared with other pages, making it worth reviewing, but this does not prove that a CTR fix will improve clicks.
* **WEAK_POSITION** — the page has a relatively poor March average search position, indicating weaker search visibility.
* **MULTI_SIGNAL** — the page shows more than one of the above conditions and therefore receives stronger review priority.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I convert the three confirmed March signals into percentile-based directional scores. Higher impressions increase priority because the page has more search exposure. Lower CTR increases priority because it represents a possible click opportunity. Worse average position increases priority because it represents weaker search visibility.

The three components are combined with equal weight. This is a baseline prioritization rule, not a predictive model. The score is used to decide which pages should be reviewed first.

In [2]:
# Check which dataframes currently exist
print([name for name in globals() if isinstance(globals()[name], pd.DataFrame)])

[]


In [1]:
# ML-07 — Section 2: Build the ranked queue

import numpy as np
import pandas as pd
from pathlib import Path

# Work from the March feature frame created earlier.
score_df = features[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
    ]
].copy()

# Make sure numeric fields are numeric.
score_df["gsc_impressions_march"] = pd.to_numeric(
    score_df["gsc_impressions_march"], errors="coerce"
)

score_df["gsc_ctr_march"] = pd.to_numeric(
    score_df["gsc_ctr_march"], errors="coerce"
)

score_df["gsc_avg_position_march"] = pd.to_numeric(
    score_df["gsc_avg_position_march"], errors="coerce"
)

# Remove rows where the three baseline signals cannot be evaluated.
score_df = score_df.dropna(
    subset=[
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
    ]
).copy()

# Keep only valid non-negative search signals.
score_df = score_df[
    (score_df["gsc_impressions_march"] >= 0)
    & (score_df["gsc_ctr_march"] >= 0)
    & (score_df["gsc_avg_position_march"] > 0)
].copy()


# ---------------------------------------------------------
# Convert each signal to a 0–1 percentile-style score
# ---------------------------------------------------------

# Higher impressions = higher priority.
score_df["exposure_score"] = (
    score_df["gsc_impressions_march"]
    .rank(method="average", pct=True)
)

# Lower CTR = higher priority.
score_df["low_ctr_score"] = 1 - (
    score_df["gsc_ctr_march"]
    .rank(method="average", pct=True)
)

# Worse average position = higher priority.
score_df["position_score"] = (
    score_df["gsc_avg_position_march"]
    .rank(method="average", pct=True)
)


# ---------------------------------------------------------
# Equal-weight directional baseline score
# ---------------------------------------------------------

score_df["baseline_action_score"] = 100 * (
    0.3333 * score_df["exposure_score"]
    + 0.3333 * score_df["low_ctr_score"]
    + 0.3334 * score_df["position_score"]
)


# ---------------------------------------------------------
# Reason-code conditions
# ---------------------------------------------------------

high_exposure_cutoff = score_df["gsc_impressions_march"].quantile(0.75)

low_ctr_cutoff = score_df["gsc_ctr_march"].quantile(0.25)

weak_position_cutoff = score_df["gsc_avg_position_march"].quantile(0.75)


score_df["high_exposure"] = (
    score_df["gsc_impressions_march"] >= high_exposure_cutoff
)

score_df["low_ctr_opportunity"] = (
    score_df["gsc_ctr_march"] <= low_ctr_cutoff
)

score_df["weak_position"] = (
    score_df["gsc_avg_position_march"] >= weak_position_cutoff
)


def make_reason_code(row):
    reasons = []

    if row["high_exposure"]:
        reasons.append("HIGH_EXPOSURE")

    if row["low_ctr_opportunity"]:
        reasons.append("LOW_CTR_OPPORTUNITY")

    if row["weak_position"]:
        reasons.append("WEAK_POSITION")

    if len(reasons) >= 2:
        return "MULTI_SIGNAL"

    if len(reasons) == 1:
        return reasons[0]

    return "NO_PRIMARY_REASON"


score_df["reason_code"] = score_df.apply(make_reason_code, axis=1)


# ---------------------------------------------------------
# Rank the complete queue
# ---------------------------------------------------------

score_df = score_df.sort_values(
    by=["baseline_action_score", "gsc_impressions_march"],
    ascending=[False, False],
).reset_index(drop=True)

score_df["rank"] = np.arange(1, len(score_df) + 1)


# ---------------------------------------------------------
# Final output columns
# ---------------------------------------------------------

baseline_queue = score_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "gsc_impressions_march",
        "gsc_ctr_march",
        "gsc_avg_position_march",
    ]
].copy()


# ---------------------------------------------------------
# Write required CSV
# ---------------------------------------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("Rows ranked:", len(baseline_queue))
print("Output written to:", output_path)

display(baseline_queue.head(20))

NameError: name 'features' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.